# How does this model compare? CLIP, SigLIP and published RSTPReid methods

Three comparisons, all on data the fine-tuned model never trained on:

1. **RSTPReid, protocol A, measured here.** CLIP ViT-B/16, CLIP ViT-L/14, SigLIP B/16 and EmbeddingGemma-2 are all run zero-shot, next to the fine-tuned model, with the same 2,000 test descriptions, 1,000 test photos and `tbpr.retrieve` code.
2. **RSTPReid, published results.** These are the numbers that well-known supervised methods (IRRA, RDE, RaSa, APTM, TBPS-CLIP, …) report on the same standard test split, copied from their papers.
3. **Video, Mall dataset.** The YOLO26s + ByteTrack tracks are computed once and shared, so every embedder sees exactly the same people. Each embedder then runs the demo's scoring (re-encode every 10 frames, EMA 0.3, box height/width ≥ 1.2) for four descriptions. The people who fit each description were labelled by eye, by track id.

## 1 · Setup

In [1]:
import json
import time
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score
from transformers import AutoModel, AutoProcessor

from tbpr import CaptionTable, Record, flatten_captions, load_records, retrieve, summarize

DATA_ROOT: Path = Path("RSTPReid")
MALL_FRAMES: Path = Path("mall_dataset/frames")
OUT: Path = Path("outputs/compare")
OUT.mkdir(parents=True, exist_ok=True)
QUERY_PROMPT: str = "SearchQuery"
VISION_TOKENS: int = 70

/home/super/Desktop/Advanced AI Models/Gemma_ebmedding_2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 · Encoders
Each model is wrapped behind the same two calls, `text(list[str])` and `images(list[Image])`, and both return L2-normalized vectors. CLIP and SigLIP use their own processors: CLIP truncates text at 77 tokens, and SigLIP pads it to 64 tokens. EmbeddingGemma-2 uses the project settings, which are bf16, the `SearchQuery` prompt for text, and 70 vision tokens.

In [2]:
class HFDualEncoder:
    """Connector to a Hugging Face CLIP-style dual encoder (separate text and image towers)."""

    def __init__(self, model_id: str, max_text_tokens: int, pad_to_max: bool) -> None:
        self.model = AutoModel.from_pretrained(model_id, torch_dtype=torch.float16).cuda().eval()
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.max_text_tokens = max_text_tokens
        self.padding: str | bool = "max_length" if pad_to_max else True

    @torch.no_grad()
    def text(self, texts: Sequence[str]) -> np.ndarray:
        out: list[torch.Tensor] = []
        for s in range(0, len(texts), 256):
            x = self.processor(text=list(texts[s:s + 256]), return_tensors="pt", padding=self.padding, truncation=True,
                               max_length=self.max_text_tokens).to("cuda")
            out.append(torch.nn.functional.normalize(self.model.get_text_features(**x).pooler_output.float(), dim=-1))
        return torch.cat(out).cpu().numpy()

    @torch.no_grad()
    def images(self, images: Sequence[Image.Image]) -> np.ndarray:
        out: list[torch.Tensor] = []
        for s in range(0, len(images), 128):
            x = self.processor(images=list(images[s:s + 128]), return_tensors="pt").to("cuda")
            feats = self.model.get_image_features(pixel_values=x["pixel_values"].half()).pooler_output.float()
            out.append(torch.nn.functional.normalize(feats, dim=-1))
        return torch.cat(out).cpu().numpy()


class GemmaEncoder:
    """Connector to EmbeddingGemma-2 through Sentence Transformers, with the project's settings."""

    def __init__(self, model_id: str) -> None:
        self.model = SentenceTransformer(model_id, device="cuda", model_kwargs={"torch_dtype": torch.bfloat16},
                                         config_kwargs={"audio_config": None})
        self.model[0].processor.image_processor.max_soft_tokens = VISION_TOKENS

    def text(self, texts: Sequence[str]) -> np.ndarray:
        return self.model.encode(list(texts), prompt_name=QUERY_PROMPT, normalize_embeddings=True, batch_size=64,
                                 show_progress_bar=False)

    def images(self, images: Sequence[Image.Image]) -> np.ndarray:
        return self.model.encode(list(images), normalize_embeddings=True, batch_size=32, show_progress_bar=False)


Encoder = HFDualEncoder | GemmaEncoder
MODELS: dict[str, tuple[str, str]] = {      # display name -> (loader kind, model id)
    "CLIP ViT-B/16": ("clip", "openai/clip-vit-base-patch16"),
    "CLIP ViT-L/14": ("clip", "openai/clip-vit-large-patch14"),
    "SigLIP B/16": ("siglip", "google/siglip-base-patch16-224"),
    "EmbeddingGemma-2 zero-shot": ("gemma", "google/embeddinggemma-2"),
    "EmbeddingGemma-2 fine-tuned (ours)": ("gemma", "models/embeddinggemma2-rstpreid"),
}


def load_encoder(kind: str, model_id: str) -> Encoder:
    if kind == "clip":
        return HFDualEncoder(model_id, 77, False)
    if kind == "siglip":
        return HFDualEncoder(model_id, 64, True)
    if kind == "gemma":
        return GemmaEncoder(model_id)
    raise ValueError(f"unknown encoder kind {kind!r} for {model_id}")


def release(enc: Encoder) -> None:
    del enc
    torch.cuda.empty_cache()

## 3 · RSTPReid, protocol A, measured here

In [3]:
records: list[Record] = load_records(DATA_ROOT / "data_captions.json")
captions: CaptionTable = flatten_captions(records)
test_rows: list[int] = [i for i, r in enumerate(records) if r.split == "test"]
q_rows: np.ndarray = np.flatnonzero(captions.split == "test")
q_texts: list[str] = [captions.texts[i] for i in q_rows]
q_pids: np.ndarray = captions.pids[q_rows]
g_pids: np.ndarray = np.array([records[i].pid for i in test_rows])
g_images: list[Image.Image] = [Image.open(DATA_ROOT / "imgs" / records[i].img_path).convert("RGB") for i in test_rows]
print(f"{len(q_texts)} queries x {len(g_images)} photos, {len(set(g_pids))} people")

measured: list[dict[str, str | float]] = []
for name, (kind, model_id) in MODELS.items():
    enc = load_encoder(kind, model_id)
    t = torch.from_numpy(enc.text(q_texts)).cuda()
    g = torch.from_numpy(enc.images(g_images)).cuda()
    measured.append({"method": name, **summarize(retrieve(t, q_pids, g, g_pids, t.shape[1], 512))})
    release(enc)
measured_df = pd.DataFrame(measured)
measured_df.to_csv(OUT / "rstpreid_measured.csv", index=False)
measured_df.round(2)

2000 queries x 1000 photos, 200 people


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 8575.81it/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights:  67%|██████▋   | 398/590 [00:00<00:00, 3914.24it/s]

... (6 outputs left out)

Loading weights: 100%|██████████| 624/624 [00:00<00:00, 10798.06it/s]

Loading weights:   0%|          | 0/624 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 624/624 [00:00<00:00, 11624.61it/s]

,method,Rank-1,Rank-5,Rank-10,mAP,mINP
0,CLIP ViT-B/16,9.85,24.65,34.40,7.20,1.64
1,CLIP ViT-L/14,11.25,26.35,37.65,8.21,2.01
2,SigLIP B/16,30.50,56.40,67.00,21.99,6.50
3,EmbeddingGemma-2 zero-shot,39.95,65.45,75.30,28.75,9.66
4,EmbeddingGemma-2 fine-tuned (ours),71.10,87.35,93.20,54.74,29.45


## 4 · RSTPReid, published results
These are the RSTPReid test-split numbers as printed in each paper. "–" means the paper does not report that metric. Every one of these methods trains on the RSTPReid train split, as the fine-tuned model here does.

Sources:
- DSSL, SSAN, IVT, CFine, IRRA, RaSa, TBPS-CLIP and RDE: Table 6 of the RDE paper ([arXiv 2308.09911](https://arxiv.org/abs/2308.09911)).
- APTM: Table 4 of [arXiv 2306.02898](https://arxiv.org/abs/2306.02898).
- The fine-tuned CLIP ViT-B/16 baseline: Table 8 of the TBPS-CLIP paper ([arXiv 2308.10045](https://arxiv.org/abs/2308.10045)).

In [4]:
@dataclass(frozen=True)
class Published:
    method: str
    venue: str
    image_backbone: str
    rank1: float
    rank5: float
    rank10: float
    map: float | None
    minp: float | None


PUBLISHED: list[Published] = [
    Published("DSSL", "ACM MM 2021", "ResNet-50", 39.05, 62.60, 73.95, None, None),
    Published("SSAN", "arXiv 2021", "ResNet-50", 43.50, 67.80, 77.15, None, None),
    Published("IVT", "ECCV-W 2022", "ViT-B/16", 46.70, 70.00, 78.80, None, None),
    Published("CFine", "TIP 2023", "CLIP ViT-B/16", 50.55, 72.50, 81.60, None, None),
    Published("CLIP fine-tuned (TBPS-CLIP baseline)", "AAAI 2024", "CLIP ViT-B/16", 56.15, 78.30, 86.60, 43.26, None),
    Published("IRRA", "CVPR 2023", "CLIP ViT-B/16", 60.20, 81.30, 88.20, 47.17, 25.28),
    Published("TBPS-CLIP", "AAAI 2024", "CLIP ViT-B/16", 61.95, 83.55, 88.75, 48.26, None),
    Published("RDE", "CVPR 2024", "CLIP ViT-B/16", 65.35, 83.95, 89.90, 50.88, 28.08),
    Published("RaSa", "IJCAI 2023", "ALBEF ViT-B/16", 66.90, 86.50, 91.35, 52.31, None),
    Published("APTM", "ACM MM 2023", "Swin-B", 67.50, 85.70, 91.45, 52.56, None),
]
ours = measured_df.set_index("method").loc["EmbeddingGemma-2 fine-tuned (ours)"]
rows = [{"method": p.method, "venue": p.venue, "image backbone": p.image_backbone, "trained on RSTPReid": "yes",
         "Rank-1": p.rank1, "Rank-5": p.rank5, "Rank-10": p.rank10, "mAP": p.map, "mINP": p.minp} for p in PUBLISHED]
for name in MODELS:
    m = measured_df.set_index("method").loc[name]
    rows.append({"method": name, "venue": "measured here", "image backbone": name.split(" (")[0],
                 "trained on RSTPReid": "yes" if "fine-tuned" in name else "no (zero-shot)",
                 "Rank-1": m["Rank-1"], "Rank-5": m["Rank-5"], "Rank-10": m["Rank-10"], "mAP": m["mAP"], "mINP": m["mINP"]})
table = pd.DataFrame(rows).sort_values("Rank-1", ascending=False).reset_index(drop=True)
table.to_csv(OUT / "rstpreid_comparison.csv", index=False)
table.round(2)

,method,venue,image backbone,trained on RSTPReid,Rank-1,Rank-5,Rank-10,mAP,mINP
0,EmbeddingGemma-2 fine-tuned (ours),measured here,EmbeddingGemma-2 fine-tuned,yes,71.10,87.35,93.20,54.74,29.45
1,APTM,ACM MM 2023,Swin-B,yes,67.50,85.70,91.45,52.56,NaN
2,RaSa,IJCAI 2023,ALBEF ViT-B/16,yes,66.90,86.50,91.35,52.31,NaN
3,RDE,CVPR 2024,CLIP ViT-B/16,yes,65.35,83.95,89.90,50.88,28.08
4,TBPS-CLIP,AAAI 2024,CLIP ViT-B/16,yes,61.95,83.55,88.75,48.26,NaN
5,IRRA,CVPR 2023,CLIP ViT-B/16,yes,60.20,81.30,88.20,47.17,25.28
6,CLIP fine-tuned (TBPS-CLIP baseline),AAAI 2024,CLIP ViT-B/16,yes,56.15,78.30,86.60,43.26,NaN
7,CFine,TIP 2023,CLIP ViT-B/16,yes,50.55,72.50,81.60,NaN,NaN
8,IVT,ECCV-W 2022,ViT-B/16,yes,46.70,70.00,78.80,NaN,NaN
9,SSAN,arXiv 2021,ResNet-50,yes,43.50,67.80,77.15,NaN,NaN


## 5 · Video: the Mall dataset
`outputs/compare/mall_tracks.json` holds the tracks: for each of the 2,000 frames, a list of `[track_id, x1, y1, x2, y2]`. They come from YOLO26s (person class, confidence 0.4, 640 px) and ByteTrack, run once, so that every embedder scores the same people.

**Labels.** For each description, the track ids of everyone who fits it were labelled by eye. The candidates were the 7 highest-scoring tracks of every model, pooled. Several shoppers can fit one description, for example two different men in red-and-white striped football shirts.

**Metrics.**
- **Top-1:** over the frames where at least one labelled person is visible and full-body shaped, the share of frames in which the best-scoring person is one of them.
- **AUC:** the probability that a labelled person outscores a non-matching person, over all frames. A high AUC means a single fixed threshold can separate them.

Both metrics are threshold-free, so models with different score scales can be compared. The last column, **crops/s**, is the image-encoding throughput on the RTX 3090.

In [5]:
QUERIES: list[str] = [
    "a man in a red and white striped shirt",
    "a man in a red t-shirt and dark jeans",
    "a woman in a red coat",
    "a woman in a light pink top and black trousers",
]
TARGETS: list[set[int]] = [                       # track ids labelled as fitting each description
    {3303, 3650, 3755, 4290, 8630, 8643},
    {3184, 3651, 5856, 7620, 9095, 9105, 9197},
    {2980, 3064, 3519, 5793, 6548, 6572, 6586, 7403, 7965},
    {5578, 9047},
]
REFRESH, EMA, PAD, MIN_ASPECT = 10, 0.3, 0.05, 1.2   # the demo's defaults (demo/track_person.py)

Box = tuple[int, int, int, int]
tracks: list[list[tuple[int, Box]]] = [[(r[0], (r[1], r[2], r[3], r[4])) for r in f]
                                       for f in json.loads((OUT / "mall_tracks.json").read_text())]
print(f"{len(tracks)} frames, {len({t for f in tracks for t, _ in f})} tracks")


def crop(frame: np.ndarray, box: Box) -> Image.Image:
    x1, y1, x2, y2 = box
    dx, dy = int((x2 - x1) * PAD), int((y2 - y1) * PAD)
    h, w = frame.shape[:2]
    patch = frame[max(0, y1 - dy):min(h, y2 + dy), max(0, x1 - dx):min(w, x2 + dx)]
    return Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))


@dataclass(frozen=True)
class FrameScores:
    scores: list[dict[int, float]]     # per query: eligible track id -> cosine score


def score_video(enc: Encoder) -> tuple[list[FrameScores], float]:
    """Run the demo's track scoring over all frames; returns per-frame scores and image crops encoded per second."""
    q = enc.text(QUERIES)
    state: dict[int, np.ndarray] = {}
    last: dict[int, int] = {}
    out: list[FrameScores] = []
    enc_s, n_enc = 0.0, 0
    for i, dets in enumerate(tracks):
        frame = cv2.imread(str(MALL_FRAMES / f"seq_{i + 1:06d}.jpg"))
        due = [(t, b) for t, b in dets if t not in state or i - last[t] >= REFRESH]
        if due:
            t0 = time.perf_counter()
            embs = enc.images([crop(frame, b) for _, b in due])
            torch.cuda.synchronize()
            enc_s, n_enc = enc_s + time.perf_counter() - t0, n_enc + len(due)
            for (t, _), e in zip(due, embs):
                mixed = e if t not in state else (1 - EMA) * state[t] + EMA * e
                state[t], last[t] = mixed / np.linalg.norm(mixed), i
        eligible = [t for t, b in dets if (b[3] - b[1]) >= MIN_ASPECT * (b[2] - b[0])]
        out.append(FrameScores([{t: float(state[t] @ q[k]) for t in eligible} for k in range(len(QUERIES))]))
    return out, n_enc / enc_s


def video_metrics(frames: list[FrameScores]) -> dict[str, float]:
    row: dict[str, float] = {}
    hits = total = 0
    aucs: list[float] = []
    for k, (query, target) in enumerate(zip(QUERIES, TARGETS)):
        y: list[bool] = []
        s: list[float] = []
        h = n = 0
        for f in frames:
            sc = f.scores[k]
            y += [t in target for t in sc]
            s += list(sc.values())
            if sc and set(sc) & target:
                n += 1
                h += max(sc, key=sc.get) in target
        row[f"top-1 · {query}"] = 100 * h / n
        hits, total = hits + h, total + n
        aucs.append(roc_auc_score(y, s))
    row["top-1 (all)"] = 100 * hits / total
    row["AUC (mean)"] = float(np.mean(aucs))
    return row


video_rows: list[dict[str, str | float]] = []
for name, (kind, model_id) in MODELS.items():
    enc = load_encoder(kind, model_id)
    frames, crops_per_s = score_video(enc)
    video_rows.append({"method": name, **video_metrics(frames), "crops/s": crops_per_s})
    release(enc)
video_df = pd.DataFrame(video_rows)
video_df.to_csv(OUT / "mall_video_comparison.csv", index=False)
video_df.round(3)

2000 frames, 1468 tracks


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 8504.21it/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights:  73%|███████▎  | 432/590 [00:00<00:00, 4316.33it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 3888.47it/s]

... (3 outputs left out)

Loading weights: 100%|██████████| 624/624 [00:00<00:00, 10903.19it/s]

Loading weights:   0%|          | 0/624 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 624/624 [00:00<00:00, 11343.18it/s]

,method,top-1 · a man in a red and white striped shirt,top-1 · a man in a red t-shirt and dark jeans,top-1 · a woman in a red coat,top-1 · a woman in a light pink top and black trousers,top-1 (all),AUC (mean),crops/s
0,CLIP ViT-B/16,86.364,100.000,81.402,100.000,84.293,0.949,344.685
1,CLIP ViT-L/14,87.662,94.444,78.167,100.000,82.373,0.936,143.326
2,SigLIP B/16,64.286,88.889,79.515,60.000,74.695,0.916,345.192
3,EmbeddingGemma-2 zero-shot,93.506,72.222,82.749,80.000,85.166,0.935,33.786
4,EmbeddingGemma-2 fine-tuned (ours),96.753,94.444,80.863,93.333,86.213,0.952,34.108


## 6 · Findings

**RSTPReid (long, detailed descriptions).** The fine-tuned model reaches **71.1 Rank-1** in this run (71.4 in `TBPR_finetune.ipynb`; bf16 re-encoding moves it by about ±0.3). That is above every published method listed here, including APTM (67.5), RaSa (66.9) and RDE (65.4), and its mINP of 29.5 is also the highest reported (RDE: 28.1, IRRA: 25.3). There are three caveats:
- EmbeddingGemma-2 is larger than the ViT-B/16 backbones of most published methods, at 868 MB of bf16 weights.
- Its pre-training data is not public.
- The published numbers are copied from the papers, not re-run here.

Zero-shot, general image–text models are weak on this task: CLIP gets 9.9 to 11.3 Rank-1 and SigLIP 30.5. RSTPReid descriptions run to 2–3 sentences and many people wear similar dark clothes, so the detail in the text is what matters.

**Video (short colour descriptions in the Mall crowd).** The gap almost closes. The fine-tuned model is best overall, with 86.2% top-1 and an AUC of 0.952. CLIP ViT-B/16, though, is close behind at 84.3% and 0.949, and it encodes crops 10× faster (345 against 34 per second).
- Short colour cues such as "red coat" are what CLIP was trained on, so CLIP handles them well.
- The fine-tuned model's clearest win is the pattern query: on "red and white striped shirt" it scores 96.8% top-1, against 86–88% for CLIP.

This benchmark is small: 4 descriptions, 573 frames with a visible target, and labels made by eye. It shows the trend rather than a precise ranking.